# ローカルで完結する対話アプリの作成

## 参考

- [Reachy Mini goes fully local (May 27, 2026, Hugging Face)](https://huggingface.co/blog/local-reachy-mini-conversation)
    - [huggingface/speech-to-speech](https://github.com/huggingface/speech-to-speech)
- [NVIDIA brings agents to life with DGX Spark and Reachy Mini (Jan 5 2026, Hugging Face)](https://huggingface.co/blog/nvidia-reachy-mini)
    - [brevdev/reachy-personal-assistant](https://github.com/brevdev/reachy-personal-assistant)
- [Building a Fully Local Voice AI Agent on a Reachy Mini Robot (April 6 2026, Hugging Face)](https://huggingface.co/blog/curtburk/reachy-voice-agent)

## 概要

ローカル環境で完結する対話アプリを作成する。このノートブックでは、Speech to Speechのカスケードパイプラインを実装する。パイプラインは4つのコンポーネントで構成される。

1. VAD（Voice Activity Detection, 音声区間検出）
1. STT （Speech to Text, 音声認識）
1. LM（Language Model, 大規模言語モデル）
1. TTS（Text to Speech, 音声合成）

## ローカルで完結させるメリット

ホスト型のリアルタイム・バックエンドは便利だが、独自のエンジンを運用することで以下のメリットが得られる。

| メリット | 説明 |
|----------|------|
| **プライバシー** | 音声データがネットワークの外に出ることがなく、パイプライン全体を自身が管理するハードウェア上で実行できる。 |
| **APIコストゼロ** | 従量課金やトークンごとの費用が発生しない。 |
| **パイプラインの完全な制御** | VAD・STT・LLM・TTS のすべてのパーツを入れ替え可能。Hugging Face Hu にさらに優れたモデルが登場した際、いつでも変更できる。 |
| **デプロイの信頼性** | クラウド依存のデモはネットワーク環境で失敗することがある。ホテルのWi-FiはAPIコールを制限したり、企業ネットワークは外部の推論エンドポイントへの通信を遮断することがある。ローカルハードウェアで動作する隔離されたシステムは、ネットワーク環境に左右されず安定して動く。 |

## ハードウェア

- Jetson Nano Orin Super
- HP ZGX Nano

## ローカルで言語モデルを動かす

```sh
# macOS
brew install llama.cpp

# windows
winget install llama.cpp
```

In [ ]:
# Gemma 4を起動。4GB程度のメモリを消費する。
!llama-server -hf ggml-org/gemma-4-E4B-it-GGUF -np 2 -c 65536 -fa on --swa-full

## Speech To Speechを動かす

In [ ]:
if not os.path.exists("speech-to-speech"):
    !git clone https://github.com/huggingface/speech-to-speech.git

In [ ]:
%pip install -e "speech-to-speech"

In [ ]:
!speech-to-speech --local_mac_optimal_settings